# 04 — Differential Abundance of Taxa and Gut Metabolic Modules (SGA vs AGA and FGR vs AGA)

**Manuscript:** Associations between gut microbiome diversity in early and late pregnancy and small for gestational age or growth restricted births: a nested case-control study

**Purpose:** Tests associations between (i) individual species (MetaPhlAn 4 relative abundances) and (ii) Gut Metabolic Modules (GMMs; HUMAnN3 KO abundances processed with Omixer-RPM) and birthweight category using MaAsLin 3, with features as outcomes. Compares SGA vs AGA and FGR vs AGA at TP1 and TP2. Models are run crude (group only) and confounder-adjusted (advanced maternal age, pre-pregnancy BMI, alcohol 3 months before pregnancy, parity, PMOS, insufficient gestational weight gain, antihypertensive use during pregnancy, and sequencing read depth, plus preeclampsia at TP2). Taxa and GMMs are analyzed in separate MaAsLin 3 runs with a shared model specification, so FDR correction is applied within each feature set. Associations with qval_individual < 0.1 are considered significant. We report both MaAsLin 3 abundance (linear, log2) and prevalence (logistic) models.

**Produces:** Supplementary Data S3: full MaAsLin 3 results for taxa (8 analyses, including covariate associations),
Supplementary Data S4: full MaAsLin 3 results for GMMs (8 analyses, including covariate associations), Figure S3:  MaAsLin 3 results for GMMs FGR vs AGA at TP1 and TP2, and SGA vs AGA at TP1 and TP2. 


**Programmer:** Anusha Antony

In [ ]:
library(dplyr)
library(tidyverse)
library(vegan)
library(ggplot2)
library(stringr)
library(maaslin3)

In [ ]:
# Define base path 
base_path_1 <- "data"
base_path_2 <- "data"
base_path_3 <- "data"

# Differential Abundance of Taxa

## SGA vs AGA

In [ ]:
sga_tp1 <- read.csv(file.path(base_path_1,"data"))
sga_tp2 <- read.csv(file.path(base_path_1, "data"))

In [ ]:
# ensure factors exist and set reference explicitly
sga_tp1$Age_above_35 <- factor(sga_tp1$Age_above_35, levels = c("0","1"))

sga_tp1$ATA_BMI_group_prior <- factor(sga_tp1$ATA_BMI_group_prior,
                                          levels = c("Normalweight","Underweight","Overweight","Obese"))
sga_tp1$Primipara <- factor(sga_tp1$Primipara, levels = c("0","1"))

sga_tp1$Q1_X62_Alcohol_3_months_before_pregnancy <- factor(sga_tp1$Q1_X62_Alcohol_3_months_before_pregnancy,
                                                               levels = c("0","1"))
sga_tp1$Q1_X31_Diagnosed_PCOS <- factor(sga_tp1$Q1_X31_Diagnosed_PCOS, levels = c("0","1"))

sga_tp1$Insufficient_weight_gain <- factor(sga_tp1$Insufficient_weight_gain, levels = c("No","Yes"))

sga_tp1$Q2_X15_during_bloodpressure_medication <- factor(sga_tp1$Q2_X15_during_bloodpressure_medication,levels = c("0","1"))  
sga_tp1$Q1_X77_during_bloodpressure_medication <- factor(sga_tp1$Q1_X77_during_bloodpressure_medication,levels = c("0","1"))                                                           

sga_tp1$preeclampsia_health_reg_ATA <- factor(sga_tp1$preeclampsia_health_reg_ATA, levels = c("No","Yes"))
sga_tp1$Group <- factor(sga_tp1$Group ,levels = c("Control","Case-SGA"))


In [ ]:
sga_tp1$kit1.faecal_sample.barcode <- as.character(sga_tp1$kit1.faecal_sample.barcode )
sga_tp1$kit2.faecal_sample.barcode <- as.character(sga_tp1$kit2.faecal_sample.barcode )

In [ ]:
head(sga_tp1)

In [ ]:
sga_tp1_row <- sga_tp1

In [ ]:
row.names(sga_tp1_row) <- sga_tp1_row$kit1.faecal_sample.barcode
sga_tp1_row <- sga_tp1_row[ , -1]
head(sga_tp1_row)

In [ ]:
sga_tp2_final <- sga_tp1 %>% 
                        filter(kit2.faecal_sample.barcode %in% sga_tp2$kit2.faecal_sample.barcode)
head(sga_tp2_final)
dim(sga_tp2_final)

In [ ]:
sga_tp2_row <- sga_tp2_final

In [ ]:
row.names(sga_tp2_row) <- sga_tp2_row$kit2.faecal_sample.barcode
sga_tp2_row <- sga_tp2_row[ , -1]
head(sga_tp2_row)

In [ ]:
# Classified taxa rarely sum to 100% per sample, since low-abundance taxa are dropped during filtering. 
#This adds an "Other_taxa" row with the missing fraction so every sample sums to 100% for plotting and comparison.
add_other_taxa <- function(taxa_table, min_total = 100, row_name = "Other_taxa") {
  
  col_totals <- colSums(taxa_table)
  new_row <- ifelse(col_totals < min_total,
                    min_total - col_totals,
                    0)

  new_row <- setNames(new_row, names(taxa_table))
  
  # Convert to one-row data frame
  other_taxa_row <- as.data.frame(t(new_row))
  rownames(other_taxa_row) <- row_name
  
  # Add row to original table
  new_table <- rbind(taxa_table, other_taxa_row)
  
  # Return result
  return(as.data.frame(new_table))
}

In [ ]:
SGA_tp1_taxa <- read.table(file.path(base_path_2, "data"), sep= "\t",
  header      = TRUE,
  check.names = FALSE
)
SGA_tp2_taxa <- read.table(file.path(base_path_2, "data") ,sep= "\t",
  header      = TRUE,
  check.names = FALSE)

In [ ]:
SGA_tp1_taxa_full <- add_other_taxa(SGA_tp1_taxa)

SGA_tp2_taxa_full <- add_other_taxa(SGA_tp2_taxa)
head(SGA_tp1_taxa_full)

In [ ]:
SGA_tp1_taxa_full<- as.data.frame(t(SGA_tp1_taxa_full))
SGA_tp2_taxa_full<- as.data.frame(t(SGA_tp2_taxa_full))

In [ ]:
folder_path_1 <- "data"

files_1 <- list.files(folder_path_1, pattern = "\\.txt$", full.names = TRUE)

read_depth <- files_1 %>%
    # tab-separated by default
  lapply(read.delim) %>%   
  bind_rows()
head(read_depth)

In [ ]:
read_depth_1<- read_depth %>%
  mutate(
    Sample_ID   = sub(".*__(\\d+)__.*", "\\1", read_depth$Sample),
    Flowcell_ID = sub(".*__(E\\d+).*", "\\1", read_depth$Sample)
  )
head(read_depth_1)

In [ ]:
read_depth_2 <- read_depth_1 %>% filter(Sample_ID %in% rownames(sga_tp1_row)) %>% 
                                        select(c(Sample_ID, Flowcell_ID, after_kraken2_host_removal))
                                  
head(read_depth_2)
dim(read_depth_2)

In [ ]:
read_depth_22 <- read_depth_1 %>% filter(Sample_ID %in% rownames(sga_tp2_row)) %>% 
                                        select(c(Sample_ID, Flowcell_ID, after_kraken2_host_removal))
                                  
head(read_depth_22)
dim(read_depth_22)

In [ ]:
read_depth_3 <- read_depth_2 %>% rename(read_depth = after_kraken2_host_removal)
head(read_depth_3)

In [ ]:
read_depth_33 <- read_depth_22 %>% rename(read_depth = after_kraken2_host_removal)
head(read_depth_33)

In [ ]:
sga_tp1_row <- sga_tp1_row %>%
  rownames_to_column(var = "Row.names")

In [ ]:
update_sga_tp1_row  <- full_join(
  sga_tp1_row ,
  read_depth_3,
  by = c("Row.names" = "Sample_ID")
)

# Put it back as row names
update_sga_tp1_row <- update_sga_tp1_row  %>%
  column_to_rownames(var = "Row.names")


In [ ]:
head(update_sga_tp1_row )

In [ ]:
SGA_tp1_taxa_full_update <- SGA_tp1_taxa_full[rownames(update_sga_tp1_row ), ]
head(SGA_tp1_taxa_full_update)

In [ ]:
sga_tp2_row <- sga_tp2_row %>%
  rownames_to_column(var = "Row.names")

In [ ]:
update_sga_tp2_row  <- full_join(
  sga_tp2_row ,
  read_depth_33,
  by = c("Row.names" = "Sample_ID")
)

# Put it back as row names
update_sga_tp2_row <- update_sga_tp2_row  %>%
  column_to_rownames(var = "Row.names")


In [ ]:
head(update_sga_tp2_row )

In [ ]:
SGA_tp2_taxa_full_update <- SGA_tp2_taxa_full[rownames(update_sga_tp2_row ), ]
head(SGA_tp2_taxa_full_update)

In [ ]:
output_SGA_TP1_crude <- file.path(base_path_1, "data/")
output_SGA_TP2_crude <- file.path(base_path_1, "data")
output_SGA_TP1_adjust <- file.path(base_path_1, "data")
output_SGA_TP2_adjust <- file.path(base_path_1, "data")

### TP1

#### Crude

In [ ]:
crude_TP1_SGA <- maaslin3(
    input_data = SGA_tp1_taxa_full_update,
    input_metadata = update_sga_tp1_row ,
    output = output_SGA_TP1_crude,
    fixed_effects = c('Group'),

    reference = c("Group", "Control"),
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH"  
)

#### Adjusted

In [ ]:
adjust_TP1_SGA <- maaslin3(
    input_data = SGA_tp1_taxa_full_update,
    input_metadata = update_sga_tp1_row,
    output = output_SGA_TP1_adjust,
    
    fixed_effects = c('Group', 'Age_above_35', 'ATA_BMI_group_prior', 'Primipara', 
                      'Q1_X62_Alcohol_3_months_before_pregnancy', 'Q1_X31_Diagnosed_PCOS',
                      'Insufficient_weight_gain', 'Q1_X77_during_bloodpressure_medication', 'read_depth'),
    

    reference = "Group,Control;Age_above_35,0;ATA_BMI_group_prior,Normalweight;Primipara,0;Q1_X62_Alcohol_3_months_before_pregnancy,0;Q1_X31_Diagnosed_PCOS,0;Insufficient_weight_gain,No;Q1_X77_during_bloodpressure_medication,0",
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH",
    plot_summary_plot = TRUE, 
    plot_associations = TRUE
    )

### TP2

#### Crude

In [ ]:
crude_TP2_SGA <- maaslin3(
    input_data = SGA_tp2_taxa_full_update,
    input_metadata = update_sga_tp2_row ,
    output = output_SGA_TP2_crude,
    fixed_effects = c('Group'),

    reference = c("Group", "Control"),
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH"  
)

#### Adjusted

In [ ]:
adjust_TP2_SGA <- maaslin3(
    input_data = SGA_tp2_taxa_full_update,
    input_metadata = update_sga_tp2_row,
    output = output_SGA_TP2_adjust,
    
    fixed_effects = c('Group', 'Age_above_35', 'ATA_BMI_group_prior', 'Primipara', 
                      'Q1_X62_Alcohol_3_months_before_pregnancy', 'Q1_X31_Diagnosed_PCOS',
                      'Insufficient_weight_gain', 'Q2_X15_during_bloodpressure_medication', 'read_depth'),
    

    reference = "Group,Control;Age_above_35,0;ATA_BMI_group_prior,Normalweight;Primipara,0;Q1_X62_Alcohol_3_months_before_pregnancy,0;Q1_X31_Diagnosed_PCOS,0;Insufficient_weight_gain,No;Q2_X15_during_bloodpressure_medication,0",
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH",
    plot_summary_plot = TRUE, 
    plot_associations = TRUE
    )

## FGR vs AGA

In [ ]:
fgr_tp1 <- read.csv(file.path(base_path_1,"data"))
fgr_tp2 <- read.csv(file.path(base_path_1, "data"))
head(fgr_tp1)
dim(fgr_tp1)

In [ ]:
# ensure factors exist and set reference explicitly
fgr_tp1$Age_above_35 <- factor(fgr_tp1$Age_above_35, levels = c("0","1"))

fgr_tp1$ATA_BMI_group_prior <- factor(fgr_tp1$ATA_BMI_group_prior,
                                          levels = c("Normalweight","Underweight","Overweight","Obese"))
fgr_tp1$Primipara <- factor(fgr_tp1$Primipara, levels = c("0","1"))

fgr_tp1$Q1_X62_Alcohol_3_months_before_pregnancy <- factor(fgr_tp1$Q1_X62_Alcohol_3_months_before_pregnancy,
                                                               levels = c("0","1"))
fgr_tp1$Q1_X31_Diagnosed_PCOS <- factor(fgr_tp1$Q1_X31_Diagnosed_PCOS, levels = c("0","1"))

fgr_tp1$Insufficient_weight_gain <- factor(fgr_tp1$Insufficient_weight_gain, levels = c("No","Yes"))

fgr_tp1$Q2_X15_during_bloodpressure_medication <- factor(fgr_tp1$Q2_X15_during_bloodpressure_medication,levels = c("0","1"))  
fgr_tp1$Q1_X77_during_bloodpressure_medication <- factor(fgr_tp1$Q1_X77_during_bloodpressure_medication,levels = c("0","1"))                                                           

fgr_tp1$preeclampsia_health_reg_ATA <- factor(fgr_tp1$preeclampsia_health_reg_ATA, levels = c("No","Yes"))
fgr_tp1$Group <- factor(fgr_tp1$Group ,levels = c("Control","Case-FGR"))


In [ ]:
fgr_tp1$kit1.faecal_sample.barcode <- as.character(fgr_tp1$kit1.faecal_sample.barcode )
fgr_tp1$kit2.faecal_sample.barcode <- as.character(fgr_tp1$kit2.faecal_sample.barcode )

In [ ]:
fgr_tp1_row <- fgr_tp1

In [ ]:
row.names(fgr_tp1_row) <- fgr_tp1_row$kit1.faecal_sample.barcode
fgr_tp1_row <- fgr_tp1_row[ , -1]
head(fgr_tp1_row)

In [ ]:
fgr_tp2_final <- fgr_tp1 %>% 
                        filter(kit2.faecal_sample.barcode %in% fgr_tp2$kit2.faecal_sample.barcode)
head(fgr_tp2_final)
dim(fgr_tp2_final)

In [ ]:
fgr_tp2_row <- fgr_tp2_final

In [ ]:
row.names(fgr_tp2_row) <- fgr_tp2_row$kit2.faecal_sample.barcode
fgr_tp2_row <- fgr_tp2_row[ , -1]
head(fgr_tp2_row)

In [ ]:
FGR_tp1_taxa <- read.table(file.path(base_path_3, "data"), sep= "\t",
  header      = TRUE,
  check.names = FALSE
)

FGR_tp2_taxa <- read.table(file.path(base_path_3, "data"), sep= "\t",
  header      = TRUE,
  check.names = FALSE
)


In [ ]:
FGR_tp1_taxa_full <- add_other_taxa(FGR_tp1_taxa)
FGR_tp2_taxa_full <- add_other_taxa(FGR_tp2_taxa)

In [ ]:
FGR_tp1_taxa_full <- as.data.frame(t(FGR_tp1_taxa_full))
FGR_tp2_taxa_full  <- as.data.frame(t(FGR_tp2_taxa_full))

In [ ]:
read_depth_fgr_2 <- read_depth_1 %>% filter(Sample_ID %in% rownames(fgr_tp1_row)) %>% 
                                        select(c(Sample_ID, Flowcell_ID, after_kraken2_host_removal))
                                  
head(read_depth_fgr_2)
dim(read_depth_fgr_2)

In [ ]:
read_depth_fgr_22 <- read_depth_1 %>% filter(Sample_ID %in% rownames(fgr_tp2_row)) %>% 
                                        select(c(Sample_ID, Flowcell_ID, after_kraken2_host_removal))
                                  
head(read_depth_fgr_22)
dim(read_depth_fgr_22)

In [ ]:
read_depth_fgr_3 <- read_depth_fgr_2 %>% rename(read_depth = after_kraken2_host_removal)
read_depth_fgr_33 <- read_depth_fgr_22 %>% rename(read_depth = after_kraken2_host_removal)


In [ ]:
fgr_tp1_row <- fgr_tp1_row %>%
  rownames_to_column(var = "Row.names")

In [ ]:
update_fgr_tp1_row  <- full_join(
  fgr_tp1_row ,
  read_depth_fgr_3,
  by = c("Row.names" = "Sample_ID")
)

# Put it back as row names
update_fgr_tp1_row <- update_fgr_tp1_row  %>%
  column_to_rownames(var = "Row.names")


In [ ]:
head(update_fgr_tp1_row )

In [ ]:
FGR_tp1_taxa_full_update <- FGR_tp1_taxa_full[rownames(update_fgr_tp1_row  ), ]
head(FGR_tp1_taxa_full_update)


In [ ]:
fgr_tp2_row <- fgr_tp2_row %>%
  rownames_to_column(var = "Row.names")

In [ ]:
update_fgr_tp2_row  <- full_join(
  fgr_tp2_row ,
  read_depth_fgr_33,
  by = c("Row.names" = "Sample_ID")
)

# Put it back as row names
update_fgr_tp2_row <- update_fgr_tp2_row  %>%
  column_to_rownames(var = "Row.names")


In [ ]:
head(update_fgr_tp2_row)

In [ ]:
FGR_tp2_taxa_full_update <- FGR_tp2_taxa_full[rownames(update_fgr_tp2_row  ), ]
head(FGR_tp2_taxa_full_update)

In [ ]:
output_FGR_TP1_crude <- file.path(base_path_1, "data")
output_FGR_TP2_crude <- file.path(base_path_1, "data")
output_FGR_TP1_adjust <- file.path(base_path_1, "data")
output_FGR_TP2_adjust <- file.path(base_path_1, "data")

### TP1

#### Crude

In [ ]:
crude_TP1_FGR <- maaslin3(
    input_data = FGR_tp1_taxa_full_update,
    input_metadata = update_fgr_tp1_row ,
    output = output_FGR_TP1_crude,
    fixed_effects = c('Group'),

    reference = c("Group", "Control"),
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH"  
)

#### Adjusted

In [ ]:
adjust_TP1_FGR<- maaslin3(
    input_data = FGR_tp1_taxa_full_update,
    input_metadata = update_fgr_tp1_row,
    output = output_FGR_TP1_adjust,
    
    fixed_effects = c('Group', 'Age_above_35', 'ATA_BMI_group_prior', 'Primipara', 
                      'Q1_X62_Alcohol_3_months_before_pregnancy', 'Q1_X31_Diagnosed_PCOS',
                      'Insufficient_weight_gain', 'Q1_X77_during_bloodpressure_medication', 'read_depth'),
    

    reference = "Group,Control;Age_above_35,0;ATA_BMI_group_prior,Normalweight;Primipara,0;Q1_X62_Alcohol_3_months_before_pregnancy,0;Q1_X31_Diagnosed_PCOS,0;Insufficient_weight_gain,No;Q1_X77_during_bloodpressure_medication,0",

    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH",
    #ADD THESE SPECIFIC PLOTTING ADJUSTMENTS: Limits the summary plot to exactly 20 species 
    plot_summary_plot = TRUE, 
    summary_plot_first_n = 20,          
    coef_plot_vars = c("Group Case-FGR"),    
    heatmap_vars = c("Group Case-FGR"),      
    plot_associations = TRUE, 
    max_pngs = 50,                    
    max_significance = 0.1 
    )

### TP2

#### Crude

In [ ]:
crude_TP2_FGR <- maaslin3(
    input_data = FGR_tp2_taxa_full_update,
    input_metadata = update_fgr_tp2_row ,
    output = output_FGR_TP2_crude,
    fixed_effects = c('Group'),

    reference = c("Group", "Control"),
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH"  
)

#### Adjusted

In [ ]:
adjust_TP2_FGR <- maaslin3(
    input_data = FGR_tp2_taxa_full_update,
    input_metadata = update_fgr_tp2_row,
    output = output_FGR_TP2_adjust,
    
    fixed_effects = c('Group', 'Age_above_35', 'ATA_BMI_group_prior', 'Primipara', 
                      'Q1_X62_Alcohol_3_months_before_pregnancy', 'Q1_X31_Diagnosed_PCOS',
                      'Insufficient_weight_gain', 'Q2_X15_during_bloodpressure_medication', 'read_depth'),
    

    reference = "Group,Control;Age_above_35,0;ATA_BMI_group_prior,Normalweight;Primipara,0;Q1_X62_Alcohol_3_months_before_pregnancy,0;Q1_X31_Diagnosed_PCOS,0;Insufficient_weight_gain,No;Q2_X15_during_bloodpressure_medication,0",
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH",
    plot_summary_plot = TRUE, 
    plot_associations = TRUE
    )

# GMM

## SGA Vs AGA

In [ ]:
module_SGA_tp1 <- read.csv(file.path(base_path_2,"data"), header = TRUE,check.names = FALSE)
head(module_SGA_tp1 )
dim(module_SGA_tp1 )

In [ ]:
module_SGA_tp2 <- read.csv(file.path(base_path_2,"data"), header = TRUE,check.names = FALSE)
head(module_SGA_tp2)
dim(module_SGA_tp2)

In [ ]:
head(update_sga_tp1_row )

In [ ]:
head(update_sga_tp2_row )

In [ ]:
module_SGA_tp1 <- module_SGA_tp1 %>% select(-c(Module))
row.names(module_SGA_tp1) <- module_SGA_tp1$Description
module_SGA_tp1 <- module_SGA_tp1[, -1]


In [ ]:
module_SGA_tp1 <- as.data.frame(t(module_SGA_tp1))
module_SGA_tp1_update <- module_SGA_tp1[rownames(update_sga_tp1_row ), ]
head(module_SGA_tp1_update)

In [ ]:
intersect(rownames(update_sga_tp2_row), rownames(update_fgr_tp2_row))

In [ ]:
identical(rownames(update_sga_tp1_row), rownames(module_SGA_tp1_update))

In [ ]:
module_SGA_tp2 <- module_SGA_tp2 %>% select(-c(Module))
row.names(module_SGA_tp2) <- module_SGA_tp2$Description
module_SGA_tp2 <- module_SGA_tp2[, -1]

In [ ]:
module_SGA_tp2 <- as.data.frame(t(module_SGA_tp2))
module_SGA_tp2_update <- module_SGA_tp2[rownames(update_sga_tp2_row ), ]
head(module_SGA_tp2_update)

In [ ]:
head(update_sga_tp2_row)

In [ ]:
identical(rownames(update_sga_tp2_row), rownames(module_SGA_tp2_update))

In [ ]:
output_SGA_TP1_crude_GMM <- file.path(base_path_1, "data")
output_SGA_TP2_crude_GMM <- file.path(base_path_1, "data")
output_SGA_TP1_adjust_GMM <- file.path(base_path_1, "data")
output_SGA_TP2_adjust_GMM <- file.path(base_path_1, "data")

### TP1

#### Crude

In [ ]:
crude_TP1_SGA_GMM <- maaslin3(
    input_data = module_SGA_tp1_update,
    input_metadata = update_sga_tp1_row ,
    output = output_SGA_TP1_crude_GMM ,
    fixed_effects = c('Group'),

    reference = c("Group", "Control"),
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH"  
)

#### Adjusted

In [ ]:
adjust_TP1_SGA_GMM <- maaslin3(
    input_data = module_SGA_tp1_update,
    input_metadata = update_sga_tp1_row,
    output = output_SGA_TP1_adjust_GMM,
    
    fixed_effects = c('Group', 'Age_above_35', 'ATA_BMI_group_prior', 'Primipara', 
                      'Q1_X62_Alcohol_3_months_before_pregnancy', 'Q1_X31_Diagnosed_PCOS',
                      'Insufficient_weight_gain', 'Q1_X77_during_bloodpressure_medication', 'read_depth'),
    

    reference = "Group,Control;Age_above_35,0;ATA_BMI_group_prior,Normalweight;Primipara,0;Q1_X62_Alcohol_3_months_before_pregnancy,0;Q1_X31_Diagnosed_PCOS,0;Insufficient_weight_gain,No;Q1_X77_during_bloodpressure_medication,0",
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH",

    
    #ADD THESE SPECIFIC PLOTTING ADJUSTMENTS: Limits the summary plot to exactly 20 species 
    plot_summary_plot = TRUE, 
    summary_plot_first_n = 20,          
    coef_plot_vars = c("Group Case-SGA"),    
    heatmap_vars = c("Group Case-SGA"),      
    plot_associations = TRUE, 
    max_pngs = 50,                    
    max_significance = 0.1
    )
    

### TP2

#### Crude

In [ ]:
crude_TP2_SGA_GMM <- maaslin3(
    input_data = module_SGA_tp2_update,
    input_metadata = update_sga_tp2_row ,
    output = output_SGA_TP2_crude_GMM ,
    fixed_effects = c('Group'),

    reference = c("Group", "Control"),
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH"  
)

#### Adjusted

In [ ]:
adjust_TP2_SGA_GMM <- maaslin3(
    input_data = module_SGA_tp2_update,
    input_metadata = update_sga_tp2_row,
    output = output_SGA_TP2_adjust_GMM,
    
    fixed_effects = c('Group', 'Age_above_35', 'ATA_BMI_group_prior', 'Primipara', 
                      'Q1_X62_Alcohol_3_months_before_pregnancy', 'Q1_X31_Diagnosed_PCOS',
                      'Insufficient_weight_gain', 'Q2_X15_during_bloodpressure_medication', 'read_depth'),
    

    reference = "Group,Control;Age_above_35,0;ATA_BMI_group_prior,Normalweight;Primipara,0;Q1_X62_Alcohol_3_months_before_pregnancy,0;Q1_X31_Diagnosed_PCOS,0;Insufficient_weight_gain,No;Q2_X15_during_bloodpressure_medication,0",
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH",

    
    #ADD THESE SPECIFIC PLOTTING ADJUSTMENTS: Limits the summary plot to exactly 20 GMM 
    plot_summary_plot = TRUE, 
    summary_plot_first_n = 20,          
    coef_plot_vars = c("Group Case-SGA"),    
    heatmap_vars = c("Group Case-SGA"),      
    plot_associations = TRUE, 
    max_pngs = 50,                    
    max_significance = 0.1
    )
    

## FGR vs AGA

In [ ]:
FGR_module_tp1 <- read.csv(file.path(base_path_3,"data"), header = TRUE,check.names = FALSE)
FGR_module_tp2 <- read.csv(file.path(base_path_3,"data"), header = TRUE,check.names = FALSE)

In [ ]:
head(update_fgr_tp1_row)
head(update_fgr_tp2_row)

In [ ]:
FGR_module_tp1 <- FGR_module_tp1 %>% select(-c(Module))
row.names(FGR_module_tp1) <- FGR_module_tp1$Description
FGR_module_tp1 <- FGR_module_tp1[, -1]


In [ ]:
FGR_module_tp2 <- FGR_module_tp2 %>% select(-c(Module))
row.names(FGR_module_tp2) <- FGR_module_tp2$Description
FGR_module_tp2 <- FGR_module_tp2[, -1]


In [ ]:
FGR_module_tp1 <- as.data.frame(t(FGR_module_tp1 ))
FGR_module_tp1_update <- FGR_module_tp1[rownames(update_fgr_tp1_row ), ]
head(FGR_module_tp1_update)

In [ ]:
FGR_module_tp2 <- as.data.frame(t(FGR_module_tp2 ))
FGR_module_tp2_update <- FGR_module_tp2[rownames(update_fgr_tp2_row ), ]
head(FGR_module_tp2_update)

In [ ]:
identical(rownames(update_fgr_tp1_row), rownames(FGR_module_tp1_update))
identical(rownames(update_fgr_tp2_row), rownames(FGR_module_tp2_update))

In [ ]:
output_FGR_TP1_crude_GMM <- file.path(base_path_1, "data")
output_FGR_TP2_crude_GMM <- file.path(base_path_1, "data")
output_FGR_TP1_adjust_GMM <- file.path(base_path_1, "data")
output_FGR_TP2_adjust_GMM <- file.path(base_path_1, "data")

### TP1

#### Crude

In [ ]:
crude_TP1_FGR_GMM <- maaslin3(
    input_data = FGR_module_tp1_update,
    input_metadata = update_fgr_tp1_row ,
    output = output_FGR_TP1_crude_GMM ,
    fixed_effects = c('Group'),

    reference = c("Group", "Control"),
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH"  
)

#### Adjusted

In [ ]:
adjust_TP1_FGR_GMM <- maaslin3(
    input_data = FGR_module_tp1_update,
    input_metadata = update_fgr_tp1_row,
    output = output_FGR_TP1_adjust_GMM ,
    
    fixed_effects = c('Group', 'Age_above_35', 'ATA_BMI_group_prior', 'Primipara', 
                      'Q1_X62_Alcohol_3_months_before_pregnancy', 'Q1_X31_Diagnosed_PCOS',
                      'Insufficient_weight_gain', 'Q1_X77_during_bloodpressure_medication', 'read_depth'),
    

    reference = "Group,Control;Age_above_35,0;ATA_BMI_group_prior,Normalweight;Primipara,0;Q1_X62_Alcohol_3_months_before_pregnancy,0;Q1_X31_Diagnosed_PCOS,0;Insufficient_weight_gain,No;Q1_X77_during_bloodpressure_medication,0",
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH",

    
    #ADD THESE SPECIFIC PLOTTING ADJUSTMENTS: Limits the summary plot to exactly 20 species 
    plot_summary_plot = TRUE, 
    summary_plot_first_n = 20,          
    coef_plot_vars = c("Group Case-FGR"),    
    heatmap_vars = c("Group Case-FGR"),      
    plot_associations = TRUE, 
    max_pngs = 50,                    
    max_significance = 0.1
    )
    

### TP2

#### Crude

In [ ]:
crude_TP2_FGR_GMM <- maaslin3(
    input_data = FGR_module_tp2_update,
    input_metadata = update_fgr_tp2_row ,
    output = output_FGR_TP2_crude_GMM  ,
    fixed_effects = c('Group'),

    reference = c("Group", "Control"),
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH"  
)

#### Adjusted

In [ ]:
adjust_TP2_FGR_GMM <- maaslin3(
    input_data = FGR_module_tp2_update,
    input_metadata = update_fgr_tp2_row,
    output = output_FGR_TP2_adjust_GMM,
    
    fixed_effects = c('Group', 'Age_above_35', 'ATA_BMI_group_prior', 'Primipara', 
                      'Q1_X62_Alcohol_3_months_before_pregnancy', 'Q1_X31_Diagnosed_PCOS',
                      'Insufficient_weight_gain', 'Q2_X15_during_bloodpressure_medication', 'read_depth'),
    

    reference = "Group,Control;Age_above_35,0;ATA_BMI_group_prior,Normalweight;Primipara,0;Q1_X62_Alcohol_3_months_before_pregnancy,0;Q1_X31_Diagnosed_PCOS,0;Insufficient_weight_gain,No;Q2_X15_during_bloodpressure_medication,0",
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH",

    
    #ADD THESE SPECIFIC PLOTTING ADJUSTMENTS: Limits the summary plot to exactly 20 GMM 
    plot_summary_plot = TRUE, 
    summary_plot_first_n = 20,          
    coef_plot_vars = c("Group Case-FGR"),    
    heatmap_vars = c("Group Case-FGR"),      
    plot_associations = TRUE, 
    max_pngs = 50,                    
    max_significance = 0.1
    )
    

# Figure S3

In [ ]:
# Point to the output folder from the maaslin3 results
fetch <- tribble(
  ~panel,                 ~dir,
  "A.  SGA vs AGA, TP1",   output_SGA_TP1_adjust_GMM,
  "B.  SGA vs AGA, TP2",   output_SGA_TP2_adjust_GMM,
  "C.  FGR vs AGA, TP1",   output_FGR_TP1_adjust_GMM,
  "D.  FGR vs AGA, TP2",   output_FGR_TP2_adjust_GMM
)

In [ ]:
# Read the group coefficient from each run
read_fetch <- function(dir, panel) {
  read_tsv(file.path(dir, "all_results.tsv"), show_col_types = FALSE) %>%
    filter(metadata == "Group") %>%
    mutate(panel = panel)
}


In [ ]:
res <- map2_dfr(fetch$dir, fetch $panel, read_fetch) %>%
  mutate(
     #lower and upper limits of the 95% confidence interval 
    low     = coef - 1.96 * stderr,
    high     = coef + 1.96 * stderr,

    fit_ok = is.na(error) & !is.na(stderr) & abs(coef) < 10,
      # "Abundance" / "Prevalence"
    model  = str_to_title(model)          
  ) %>%
  filter(fit_ok)

In [ ]:
# For the figures, sleceting only 20 top GMM's
top_gmm <- 20   
gmm_order <- res %>%
  group_by(feature) %>%
  summarise(min_q = min(qval_individual, na.rm = TRUE)) %>%
  slice_min(min_q, n = top_gmm, with_ties = FALSE) %>%
 # smallest q ends up at the top
  arrange(desc(min_q)) %>%                  
    pull(feature)


In [ ]:
#Arrange a DF         
make_df  <- res %>%
  filter(feature %in% gmm_order) %>%
  mutate(feature = factor(feature, levels = gmm_order ))

In [ ]:
dodge<- position_dodge(width = 0.6)

In [ ]:
Final_s3 <- ggplot(make_df, aes(
  x = coef, y = feature, shape = model, fill = qval_individual
)) +
  geom_vline(
    xintercept = 0,
    linetype = "dashed",
    colour = "grey50"
  ) +
  geom_errorbarh(
    aes(xmin = low, xmax = high),
    height = 0.3,
    position = dodge
  ) +
  geom_point(
    size = 2.8,
    position = dodge
  ) +
  facet_wrap(~ panel, ncol = 2) +
  scale_shape_manual(
    values = c(Abundance = 21, Prevalence = 24),
    name = "Model"
  ) +
  scale_fill_gradient(
    low = "#8B1A89",
    high = "white",
    limits = c(0, 1),
    name = expression(q[individual])
  ) +
  labs(
    x = expression(beta ~ "coefficient (95% CI)"),
    y = NULL
  ) +
  theme_bw(base_size = 10) +
  theme(
    # Axis
    axis.text.x = element_text(size = 10, colour = "black"),
    axis.text.y = element_text(size = 10, colour = "black"),
    axis.title.x = element_text(size = 11),

    # Facet labels
    strip.text = element_text(
      size = 11,
      face = "bold",
      hjust = 0
    ),
    strip.background = element_rect(fill = "grey95"),

    # Legend
    legend.text = element_text(size = 10),
    legend.title = element_text(size = 11),

    # Grid
    panel.grid.minor = element_blank(),

    # Legend position
    legend.position = "right"
  )



In [ ]:
ggsave(
  "Figure_S3_GMM_adjusted.pdf",
  Final_s3,
  width = 8.5,
  height = 11,
  device = cairo_pdf
)

In [ ]:
ggsave(
  "Figure_S3_GMM_adjusted.eps",
  Final_s3,
  width = 8.5,
  height = 11,
  device = "eps"
)

# Supplementary Data S1 and S2

In [ ]:
merge_groups <- function(SGA_df, FGR_df) {
  

  # Check for duplicate sample IDs within each dataframe
  sga_duplicates <- unique(rownames(SGA_df)[duplicated(rownames(SGA_df))])
  fgr_duplicates <- unique(rownames(FGR_df)[duplicated(rownames(FGR_df))])
  
  if (length(sga_duplicates) > 0) {
    stop(
      "Duplicate Sample IDs found in SGA: ",
      paste(sga_duplicates, collapse = ", ")
    )
  }
  
  if (length(fgr_duplicates) > 0) {
    stop(
      "Duplicate Sample IDs found in FGR: ",
      paste(fgr_duplicates, collapse = ", ")
    )
  }
  
  # Convert rownames to Sample_ID
  SGA <- SGA_df %>%
    rownames_to_column("Sample_ID") %>%
    mutate(Group = "Case-SGA")
  
  FGR <- FGR_df %>%
    rownames_to_column("Sample_ID") %>%
    mutate(Group = "Case-FGR")
  
  # Sample IDs present in both datasets
  control_ids <- intersect(SGA$Sample_ID, FGR$Sample_ID)
  
  # Overlapping samples = Control
  SGA <- SGA %>%
    mutate(
      Group = if_else(
        Sample_ID %in% control_ids,
        "Control",
        Group
      )
    )
  
  # Remove duplicate samples from FGR
  FGR <- FGR %>%
    filter(!Sample_ID %in% control_ids)
  
  # Combine
  merged_df <- bind_rows(SGA, FGR) %>%
    select(Sample_ID, Group, everything())
  
  return(merged_df)
}

In [ ]:
merged_SGA_FGR_tp1 <- merge_groups(SGA_tp1_taxa_full, FGR_tp1_taxa_full)
head(merged_SGA_FGR_tp1)

In [ ]:
merged_SGA_FGR_tp2 <- merge_groups(SGA_tp2_taxa_full, FGR_tp2_taxa_full)
head(merged_SGA_FGR_tp2)

In [ ]:
table(merged_SGA_FGR_tp2$Group)

In [ ]:
table(merged_SGA_FGR_tp1$Group)

In [ ]:
SGA_FGR_GMM_tp1 <- merge_groups(module_SGA_tp1_update, FGR_module_tp1_update)
head(SGA_FGR_GMM_tp1)

In [ ]:
SGA_FGR_GMM_tp2 <- merge_groups(module_SGA_tp2_update, FGR_module_tp2_update)
head(SGA_FGR_GMM_tp2)

In [ ]:
table(SGA_FGR_GMM_tp2$Group)
table(SGA_FGR_GMM_tp1$Group)

In [ ]:

write_tsv(merged_SGA_FGR_tp1,
  file.path(base_path_1, "data")
)

In [ ]:
write_tsv(merged_SGA_FGR_tp2,
  file.path(base_path_1, "data")
)

In [ ]:
write_tsv(SGA_FGR_GMM_tp1,
  file.path(base_path_1, "data")
)

In [ ]:
write_tsv(SGA_FGR_GMM_tp2,
  file.path(base_path_1, "data")
)